TODO add model_type == logistic regressor (for a less advanced machine-learning model, close to symbolic)
[Already compatible with GridSearch](https://fairlearn.org/main/auto_examples/plot_grid_search_census.html) and entails [sample_weights](https://stackoverflow.com/questions/46204331/how-to-set-sample-weight-in-sklearn-logistic-regression) option


TODO add selection by feature importance: [SHAP](https://towardsdatascience.com/pytorch-shap-explainable-convolutional-neural-networks-ece5f04c374f) -> find the best [method](https://stackoverflow.com/questions/70510341/shap-values-with-pytorch-kernelexplainer-vs-deepexplainer) to sort features

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

# Discrimination detection and mitigation (on revenues classification dataset)

## Train a model regardless of fairness

In [ ]:
import sys
sys.path.append("../")

from matplotlib import pyplot
import seaborn as sns

from fairdream.data_preparation import *
from fairdream.compute_scores import *
from fairdream.detection import *
from fairdream.correction import *
from fairdream.plots import *

In [ ]:
# set your statistics purposes, as the type of machine-learning model to train (tree- / neural- based)
model_task = 'classification'
stat_criteria = 'auc'

model_type = 'random_forest' # 'log_reg'# 'neural_net' # 'xgboost'

### Prepare data

Fix precise % of population distribution (sex: Male, Female) and % of loan granted according to sex, to inspect the effects of FairDream.

In [ ]:
# preparing the dataset on clients for binary classification
from sklearn.datasets import fetch_openml
data = fetch_openml(data_id=1590, as_frame=True)

X = data.data
Y = (data.target == '>50K') * 1

In [ ]:
dataset = X.copy()
dataset['target'] = Y
dataset

In [ ]:
Y

### Bring your own model 

If you want to bring your own model, you have to set 3 features:

1. uncorrected_model_path
Save your model in uncorrected_model_path, for fairness analysis on relevant features
Ex: uncorrected_model_path = "/work/data/models/uncorrected_model.pkl"

2. X_train_valid, Y_train_valid
pd.DataFrame with your inputs and targets on train&valid set, of shape(nb_individuals,)

3. Y_pred_train_valid
np.ndarray with the predicted label (i.e. class) or value, of shape(nb_individuals,)

### Automatically train a model statistically performant, regardless of fairness

In [ ]:
X_train, X_valid, X_train_valid, X_test, Y_train, Y_valid, Y_train_valid, Y_test = train_valid_test_split(X,Y, model_task)

In [ ]:
# save the uncorrected model, to then sort its features by importances
save_model=True
uncorrected_model_path = "/work/data/models/uncorrected_model.pkl"

Y_pred_train_valid = train_naive_model(X_train, X_valid, X_train_valid, X_test, Y_train, Y_valid, Y_train_valid, Y_test, model_task, stat_criteria, save_model=save_model,
                                      model_type=model_type)

## Detection alert (on train&valid data to examine if the model learned discriminant behavior)

In [ ]:
train_valid_set_with_uncorrected_results = augment_train_valid_set_with_results("uncorrected", X_train_valid, Y_train_valid, Y_pred_train_valid, model_task)
# train_valid_set_with_uncorrected_results

In [ ]:
train_valid_set_with_uncorrected_results

In [ ]:
augmented_train_valid_set = train_valid_set_with_uncorrected_results
model_name = "uncorrected"

fairness_purpose = "overall_positive_rate"
injustice_acceptance=3
min_individuals_discrimined=0.01

discrimination_alert(augmented_train_valid_set, model_name, fairness_purpose, model_task, injustice_acceptance, min_individuals_discrimined)

In [ ]:
augmented_train_valid_set = train_valid_set_with_uncorrected_results
model_name = "uncorrected"

fairness_purpose = "overall_positive_rate"
injustice_acceptance=3
min_individuals_discrimined=0.01

discrimination_alert(augmented_train_valid_set, model_name, fairness_purpose, model_task, injustice_acceptance, min_individuals_discrimined)

## Discrimination correction with a new fair model

### Generating fairer models with grid search or weights distorsion

In [ ]:
# the user determines one's fairness objectives to build new fairer models
# on which group and regarding which criteria (purpose, constraint of the models) one aims to erase discrimination

protected_attribute = "sex"

# then the user sets the desired balance between stat and fair performances 
tradeoff = "fair_preferred"
weight_method = 'weighted_groups'
nb_fair_models = 4


train_valid_set_with_corrected_results, models_df, best_model_dict = fair_train(
    X=X,
    Y=Y,
    train_valid_set_with_uncorrected_results=train_valid_set_with_uncorrected_results,
    protected_attribute=protected_attribute,
    fairness_purpose=fairness_purpose,
    model_task=model_task,
    stat_criteria=stat_criteria,
    tradeoff=tradeoff,
    weight_method=weight_method,
    nb_fair_models=nb_fair_models,
    model_type=model_type,
)

In [ ]:
# TODO plot with the ROC-AUC of baseline, FairDream and GridSearch together (best after experiment -> plot)

### Evaluating the best fair model

In [ ]:
fair_model_results(train_valid_set_with_corrected_results, models_df, best_model_dict,protected_attribute,fairness_purpose, model_task)

In [ ]:
top_models = models_df.sort_values(by='tradeoff_score',ascending=False)
top_models

In [ ]:
from fairdream.plots import plot_compared_metrics

list_compared_metrics = ['nb_positive','overall_positive_rate', 'true_positive_rate', 'true_negative_rate',
                         'false_positive_rate','false_negative_rate']

plot_compared_metrics(
                            train_valid_set_with_corrected_results=train_valid_set_with_corrected_results,
                            protected_attribute=protected_attribute,
                            fairness_purpose=fairness_purpose,
                            model_task=model_task,
                            best_model_dict=best_model_dict,
                            list_compared_metrics=list_compared_metrics,
                            )

### Plots of the new distributions

First, we plot the distribution of individuals, and wealthy individuals, in the training&valid set:

In [ ]:
list_selected=["Best model", "Not selected", "Baseline", "Not selected", "Not selected"]
list_stat_score_values=[0.878, 0.882, 0.888, 0.892, 0.898]
list_fair_score_values=[2.97, 2.77, 2.87, 2.86, 2.82]
list_tradeoff_scores=[0, 0, 0, 0, 0]

models_df = pd.DataFrame({
    "selected":list_selected,
    "stat_score_value":list_stat_score_values,
    "fair_score_value":list_fair_score_values,})

plot_all_scores(models_df)

In [ ]:
#train_valid_set_with_corrected_results=train_valid_set_with_uncorrected_results.copy()
train_valid_set_with_corrected_results['target']=train_valid_set_with_uncorrected_results['target_train_valid']

In [ ]:
df_young = train_valid_set_with_corrected_results.loc[train_valid_set_with_corrected_results['age']<29]
df_young_wealthy = df_young.loc[df_young['target']==1]
print(df_young.shape[0])
print(df_young_wealthy.shape[0])
print(df_young_wealthy.shape[0]/df_young.shape[0])

In [ ]:
df_old = train_valid_set_with_corrected_results.loc[(train_valid_set_with_corrected_results['age']>=29)
                                                    &(train_valid_set_with_corrected_results['age']<=37)]
df_old_wealthy = df_old.loc[df_old['target']==1]
print(df_old.shape[0])
print(df_old_wealthy.shape[0])
print(df_old_wealthy.shape[0]/df_old.shape[0])

In [ ]:
n_bins = train_valid_set_with_corrected_results['age'].nunique()-2

age_values, age_counts = np.unique(train_valid_set_with_corrected_results['age'], return_counts=True)
#[random.gauss(3,1) for _ in range(400)]
#income_values, income_counts = np.unique(train_valid_set_with_corrected_results['target_train_valid'], return_counts=True)
#[random.gauss(4,2) for _ in range(400)]
df_income_age = train_valid_set_with_corrected_results[['target_train_valid', 'age']]
age_counts = df_income_age['age']

df_wealthy_income_age = df_income_age.loc[df_income_age['target_train_valid']==1]
income_counts = df_wealthy_income_age['age']

bins = np.linspace(0, 90, 1)

pyplot.figure(figsize=(10, 8), dpi=80)

pyplot.hist(age_counts, alpha=0.5, label="Number of individuals per age", bins=n_bins)
pyplot.hist(income_counts, alpha=0.5, label="Number of individuals earning over $50,000 per age", bins=n_bins)

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()

In [ ]:
train_valid_set_with_corrected_results = train_valid_set_with_corrected_results.loc[train_valid_set_with_corrected_results["age"]<=37]

At a first glance, it seems there has been no progress (first branch of the Simpson's paradox)...

In [ ]:
fair_model_name = best_model_dict['model_name']

age_values, age_counts = np.unique(train_valid_set_with_corrected_results['age'], return_counts=True)
#[random.gauss(3,1) for _ in range(400)]
#income_values, income_counts = np.unique(train_valid_set_with_corrected_results['target_train_valid'], return_counts=True)
#[random.gauss(4,2) for _ in range(400)]
df_income_age = train_valid_set_with_corrected_results[['target_train_valid', 'age']]
age_counts = df_income_age['age']

# for baseline models' predictions
df_income_age_uncorrected = train_valid_set_with_corrected_results[['predicted_uncorrected', 'age']]

df_wealthy_income_age_uncorrected = df_income_age_uncorrected.loc[df_income_age_uncorrected['predicted_uncorrected']==1]
income_counts_uncorrected = df_wealthy_income_age_uncorrected['age']

# and for fair models' predictions
df_income_age_fair = train_valid_set_with_corrected_results[[f"predicted_{fair_model_name}", 'age']]

df_wealthy_income_age_fair = df_income_age_fair.loc[df_income_age_fair[f"predicted_{fair_model_name}"]==1]
income_counts_fair = df_wealthy_income_age_fair['age']

bins = np.linspace(0, 90, 1)

pyplot.figure(figsize=(10, 8), dpi=80)

pyplot.hist(income_counts_fair, alpha=0.5, label="Number of predicted 'wealthy' (fair model)",
           color='blue')

pyplot.hist(income_counts_uncorrected, alpha=0.5, label="Number of predicted 'wealthy' (baseline model)",
           color = 'red')

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()

But we have to look at the results **given** people are truly wealthy (resp. truly unwealthy)...

In [ ]:
age_values, age_counts = np.unique(train_valid_set_with_corrected_results['age'], return_counts=True)
#[random.gauss(3,1) for _ in range(400)]
#income_values, income_counts = np.unique(train_valid_set_with_corrected_results['target_train_valid'], return_counts=True)
#[random.gauss(4,2) for _ in range(400)]
df_income_age = train_valid_set_with_corrected_results[['target_train_valid', 'age']]
age_counts = df_income_age['age']

df_unwealthy_income_age = df_income_age.loc[df_income_age['target_train_valid']==0]
income_counts = df_unwealthy_income_age['age']

# and for baseline models' predictions
df_income_age_uncorrected = train_valid_set_with_corrected_results[['predicted_uncorrected', 'age']]

df_unwealthy_income_age_uncorrected = df_income_age_uncorrected.loc[df_income_age_uncorrected['predicted_uncorrected']==0]
income_counts_uncorrected = df_unwealthy_income_age_uncorrected['age']

bins = np.linspace(0, 90, 1)

pyplot.figure(figsize=(10, 8), dpi=80)

pyplot.hist(income_counts, alpha=0.5, label="Number of 'unwealthy' individuals per age",
           color="gold")
pyplot.hist(income_counts_uncorrected, alpha=0.5, label="Number of predicted 'unwealthy' (baseline model)",
           color="red")

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()

In [ ]:
age_values, age_counts = np.unique(train_valid_set_with_corrected_results['age'], return_counts=True)
#[random.gauss(3,1) for _ in range(400)]
#income_values, income_counts = np.unique(train_valid_set_with_corrected_results['target_train_valid'], return_counts=True)
#[random.gauss(4,2) for _ in range(400)]
df_income_age = train_valid_set_with_corrected_results[['target_train_valid', 'age']]
age_counts = df_income_age['age']

df_unwealthy_income_age = df_income_age.loc[df_income_age['target_train_valid']==0]
income_counts = df_unwealthy_income_age['age']

# and for fair models' predictions
df_income_age_fair = train_valid_set_with_corrected_results[[f"predicted_{fair_model_name}", 'age']]

df_unwealthy_income_age_fair = df_income_age_fair.loc[df_income_age_fair[f"predicted_{fair_model_name}"]==0]
income_counts_fair = df_unwealthy_income_age_fair['age']

bins = np.linspace(0, 90, 1)

pyplot.figure(figsize=(10, 8), dpi=80)

pyplot.hist(income_counts, alpha=0.5, label="Number of 'unwealthy' individuals per age",
           color="gold")
pyplot.hist(income_counts_fair, alpha=0.5, label="Number of predicted 'unwealthy' (fair model)",
           color="blue")

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()

And for truly wealthy predictions, we see in the same way a (maximal possible) improvement of results by young individuals:

In [ ]:
age_values, age_counts = np.unique(train_valid_set_with_corrected_results['age'], return_counts=True)
#[random.gauss(3,1) for _ in range(400)]
#income_values, income_counts = np.unique(train_valid_set_with_corrected_results['target_train_valid'], return_counts=True)
#[random.gauss(4,2) for _ in range(400)]
df_income_age = train_valid_set_with_corrected_results[['target_train_valid', 'age']]
age_counts = df_income_age['age']

df_wealthy_income_age = df_income_age.loc[df_income_age['target_train_valid']==1]
income_counts = df_wealthy_income_age['age']

# and for baseline models' predictions
df_income_age_uncorrected = train_valid_set_with_corrected_results[['predicted_uncorrected', 'age']]

df_wealthy_income_age_uncorrected = df_income_age_uncorrected.loc[df_income_age_uncorrected['predicted_uncorrected']==1]
income_counts_uncorrected = df_wealthy_income_age_uncorrected['age']

bins = np.linspace(0, 90, 1)

pyplot.figure(figsize=(10, 8), dpi=80)

pyplot.hist(income_counts_uncorrected, alpha=0.5, label="Number of predicted 'wealthy' (baseline model)")
pyplot.hist(income_counts, alpha=0.5, label="Number of 'wealthy' individuals per age")

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()

In [ ]:
age_values, age_counts = np.unique(train_valid_set_with_corrected_results['age'], return_counts=True)
#[random.gauss(3,1) for _ in range(400)]
#income_values, income_counts = np.unique(train_valid_set_with_corrected_results['target_train_valid'], return_counts=True)
#[random.gauss(4,2) for _ in range(400)]
df_income_age = train_valid_set_with_corrected_results[['target_train_valid', 'age']]
age_counts = df_income_age['age']

df_wealthy_income_age = df_income_age.loc[df_income_age['target_train_valid']==1]
income_counts = df_wealthy_income_age['age']

# and for fair models' predictions
df_income_age_fair = train_valid_set_with_corrected_results[[f"predicted_{fair_model_name}", 'age']]

df_wealthy_income_age_fair = df_income_age_fair.loc[df_income_age_fair[f"predicted_{fair_model_name}"]==1]
income_counts_fair = df_wealthy_income_age_fair['age']

bins = np.linspace(0, 90, 1)

pyplot.figure(figsize=(10, 8), dpi=80)

pyplot.hist(income_counts_fair, alpha=0.5, label="Number of predicted 'wealthy' (fair model)")
pyplot.hist(income_counts, alpha=0.5, label="Number of 'wealthy' individuals per age")

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()

In [ ]:
# first, reconstitute the dataset (distribution of true labels) == (truly wealthy, truly non-wealthy)
# old (2400, 1266)
# young (547, 3069)

age_young_max = 29
age_old_max = 37

nb_old_wealthy = 2400
nb_old_unwealthy = 1266
nb_young_wealthy = 547
nb_young_unwealthy = 3069

df_old_wealthy_new, df_old_unwealthy_new, df_young_wealthy_new, df_young_unwealthy_new = set_age_effect(
                  dataset=dataset, age_young_max=age_young_max, age_old_max=age_old_max,
                  nb_old_wealthy=nb_old_wealthy, nb_old_unwealthy=nb_old_unwealthy,
                  nb_young_wealthy=nb_young_wealthy, nb_young_unwealthy=nb_young_unwealthy)

In [ ]:
# uncorrected model

print("Results for uncorrected model \n")

# for older group
# predicted wealthy, predicted not wealthy: among true wealthy (2166, 234), among true unwealthy (253, 1013)
print("Uncorrected model for older group")
df_uncorrected_results_old = set_wealthiness_prediction(
    model_name = "uncorrected",
    df_positive=df_old_wealthy_new, df_negative=df_old_unwealthy_new, 
    nb_true_positive=2166, nb_true_negative=253)

# for older group
print("\n Uncorrected model for younger group")
df_uncorrected_results_young = set_wealthiness_prediction(
    model_name = "uncorrected",
    df_positive=df_young_wealthy_new, df_negative=df_young_unwealthy_new, 
    nb_true_positive=371, nb_true_negative=61)

In [ ]:
# New "fairer" model

print("Results for (trained to be) fairer model \n")

# for older group
print("New 'fairer' model for older group")
df_fair_results_old = set_wealthiness_prediction(
    model_name = "fair",
    df_positive=df_old_wealthy_new, df_negative=df_old_unwealthy_new, 
    nb_true_positive=2112, nb_true_negative=50)

# for older group
print("\n New 'fairer' model for younger group")
df_fair_results_young = set_wealthiness_prediction(
    model_name = "fair",
    df_positive=df_young_wealthy_new, df_negative=df_young_unwealthy_new, 
    nb_true_positive=486, nb_true_negative=92)

Then, we plot the new distribution of "predicted wealthy" individuals per age

In [ ]:
# finally, aggregate the DataFrames for comparison of predictions
df_uncorrected_results = df_uncorrected_results_old.append([df_uncorrected_results_young])
df_uncorrected_results.sort_index(inplace=True)
df_uncorrected_results.reset_index(drop=True, inplace=True)

df_fair_results = df_fair_results_old.append([df_fair_results_young])
df_fair_results.sort_index(inplace=True)
df_fair_results.reset_index(drop=True, inplace=True)

df_fair_and_uncorrected_results = df_fair_results.copy()
df_fair_and_uncorrected_results["predicted_uncorrected"] = df_uncorrected_results["predicted_uncorrected"]

df_fair_and_uncorrected_results

Plot the initial distribution (age, wealthiness):

In [ ]:
pyplot.figure(figsize=(8, 6), dpi=80)

sns.histplot(data=df_fair_and_uncorrected_results["age"],
             bins=10,
             color="gray", label="Number of individuals per age")

sns.histplot(df_fair_and_uncorrected_results.loc[df_fair_and_uncorrected_results["target"]==1]["age"],
             bins=9,
             color="gold", label="Number of 'wealthy' individuals per age")

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()

First, compare the predictions in absolute numbers ("uncorrected" vs "fair"):

In [ ]:
pyplot.figure(figsize=(8, 6), dpi=80)

sns.histplot(data=df_fair_and_uncorrected_results.loc[df_fair_and_uncorrected_results["predicted_fair"]==1]["age"], 
             bins=10,
             color="lightskyblue", label="Number of predicted 'wealthy' (fair model)")

sns.histplot(data=df_fair_and_uncorrected_results.loc[df_fair_and_uncorrected_results["predicted_uncorrected"]==1]["age"],
             bins=10,
             color="red", label="Number of predicted 'wealthy' (baseline model)")

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()

Finally, compare it according to the real distribution of labels across ages.
First, for baseline model:

In [ ]:
pyplot.figure(figsize=(8, 6), dpi=80)

sns.histplot(data=df_fair_and_uncorrected_results.loc[df_fair_and_uncorrected_results["predicted_uncorrected"]==1]["age"],
             bins=10,
             color="red", label="Number of predicted 'wealthy' (baseline model)")

sns.histplot(df_fair_and_uncorrected_results.loc[df_fair_and_uncorrected_results["target"]==1]["age"],
             bins=9,
             color="yellow", label="Number of 'wealthy' individuals per age")

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()

Second, for the (trained to be) fairer model:

In [ ]:
pyplot.figure(figsize=(8, 6), dpi=80)

sns.histplot(data=df_fair_and_uncorrected_results.loc[df_fair_and_uncorrected_results["predicted_fair"]==1]["age"], 
             bins=10,
             color="lightskyblue", label="Number of predicted 'wealthy' (fair model)")

sns.histplot(df_fair_and_uncorrected_results.loc[df_fair_and_uncorrected_results["target"]==1]["age"],
             bins=9,
             color="gold", label="Number of 'wealthy' individuals per age")

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()

### Are the new results "fairer"? To the Simpson Paradox

Now, we focus on the "younger" group (i.e. between 17-29 years old). Compared to the results considering the overall population, it seems to be no progress:

But it seems to be due to the fact that they are under-represented in the (true distribution of) wealthiness across initial data:

In [ ]:
pyplot.figure(figsize=(8, 6), dpi=80)

sns.histplot(data=df_fair_and_uncorrected_results.loc[df_fair_and_uncorrected_results["age"]<29]["age"],
             bins=12,
             color="gray", label="Number of individuals per age")

sns.histplot(df_fair_and_uncorrected_results.loc[(df_fair_and_uncorrected_results["age"]<29)
                                                 &(df_fair_and_uncorrected_results["target"]==1)]["age"],
             bins=10,
             color="gold", label="Number of 'wealthy' individuals per age")

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()

And indeed, we see that considering this imbalanced distribution of wealthiness, i.e. **given** the true wealthy people, the new model achieves the **best possible results** on young individuals.

There are the predictions "before" training for fairness (baseline model)...

In [ ]:
pyplot.figure(figsize=(8, 6), dpi=80)

sns.histplot(data=df_fair_and_uncorrected_results.loc[
    (df_fair_and_uncorrected_results["age"]<29)&(df_fair_and_uncorrected_results["predicted_uncorrected"]==1)]["age"],
             bins=12,
             color="red", label="Number of predicted 'wealthy' (baseline model)")

sns.histplot(df_fair_and_uncorrected_results.loc[(df_fair_and_uncorrected_results["age"]<29)
                                                 &(df_fair_and_uncorrected_results["target"]==1)]["age"],
             bins=10,
             color="gold", label="Number of 'wealthy' individuals per age")

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()

... And after training for fairness (new "fairer" model):

In [ ]:
pyplot.figure(figsize=(8, 6), dpi=80)

sns.histplot(data=df_fair_and_uncorrected_results.loc[
    (df_fair_and_uncorrected_results["age"]<29)&(df_fair_and_uncorrected_results["predicted_fair"]==1)]["age"],
             bins=12,
             color="lightskyblue", label="Number of predicted 'wealthy' (fairer model)")

sns.histplot(df_fair_and_uncorrected_results.loc[(df_fair_and_uncorrected_results["age"]<29)
                                                 &(df_fair_and_uncorrected_results["target"]==1)]["age"],
             bins=10,
             color="gold", label="Number of 'wealthy' individuals per age")

pyplot.xlabel('Age')
pyplot.ylabel('Number of clients')
pyplot.legend(loc='upper right')
pyplot.show()